# Day 9 - Logistic Regression

Predicting loan approval. Logistic Regression is the standard baseline for binary classification problems.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve, auc

%matplotlib inline

## 1. Load Data

In [ ]:
df = pd.read_csv('dataset/loans.csv')
print('Shape:', df.shape)
print(df['approved'].value_counts())
df.head()

## 2. Train / Test Split

In [ ]:
feature_cols = ['income', 'loan_amount', 'credit_score', 'employment_years',
                'existing_debt', 'dependents', 'loan_term_months']

X = df[feature_cols]
y = df['approved']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 3. Train Model

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
print(f'CV Accuracy: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')

## 4. Evaluation

In [ ]:
preds = model.predict(X_test_scaled)
probs = model.predict_proba(X_test_scaled)[:, 1]

print(f'Test Accuracy: {accuracy_score(y_test, preds):.4f}\n')
print(classification_report(y_test, preds, target_names=['Rejected', 'Approved']))

In [ ]:
fpr, tpr, _ = roc_curve(y_test, probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='#2196F3', linewidth=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

In [ ]:
coefficients = model.coef_[0]
sorted_idx = np.argsort(np.abs(coefficients))[::-1]

plt.figure(figsize=(9, 5))
colors_bar = ['#4CAF50' if c > 0 else '#F44336' for c in coefficients[sorted_idx]]
plt.bar([feature_cols[i] for i in sorted_idx], coefficients[sorted_idx], color=colors_bar)
plt.title('Feature Coefficients')
plt.ylabel('Coefficient')
plt.xticks(rotation=25, ha='right')
plt.axhline(y=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

# positive coefficients increase approval odds, negative ones decrease it

## Wrap-up

97.25% test accuracy with a 0.998 ROC AUC. Credit score and debt-to-income ratio were the strongest predictors.

Logistic Regression coefficients are directly interpretable - this is one of its biggest advantages over black-box models like Random Forest. You can literally explain to a loan applicant which factors hurt or helped their case.